# Phân lớp Ung thư Vú với Breast Cancer Wisconsin (Original)

**Mục tiêu:**  
Thực hiện phân lớp (predict `Class`) dự đoán ung thư vú dựa trên các biến đo tế bào, sử dụng 3 mô hình phân lớp khác nhau từ thư viện scikit-learn.

**Các bước thực hiện:**  
1. Nạp và tiền xử lý dữ liệu  
2. Khám phá dữ liệu (EDA)  
3. Xây dựng và huấn luyện mô hình  
4. Đánh giá hiệu năng mô hình

## 1. Nạp thư viện và dữ liệu

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Đọc dữ liệu
column_names = ['Sample code number', 'Clump Thickness', 'Uniformity of Cell Size',
                'Uniformity of Cell Shape', 'Marginal Adhesion',
                'Single Epithelial Cell Size', 'Bare Nuclei', 'Bland Chromatin',
                'Normal Nucleoli', 'Mitoses', 'Class']
df = pd.read_csv('breast-cancer-wisconsin.data', header=None, names=column_names)

# Bỏ cột Sample code number
df = df.drop('Sample code number', axis=1)
df.head()

,Clump Thickness,Uniformity of Cell Size,Uniformity of Cell Shape,Marginal Adhesion,Single Epithelial Cell Size,Bare Nuclei,Bland Chromatin,Normal Nucleoli,Mitoses,Class
0,5,1,1,1,2,1,3,1,1,2
1,5,4,4,5,7,10,3,2,1,2
2,3,1,1,1,2,2,3,1,1,2
3,6,8,8,1,3,4,3,7,1,2
4,4,1,1,3,2,1,3,1,1,2


## 2. Tiền xử lý dữ liệu
- Xử lý giá trị thiếu và chuyển kiểu dữ liệu đúng

In [4]:
# Nhận diện giá trị thiếu (ký hiệu '?')
df.replace('?', pd.NA, inplace=True)
df['Bare Nuclei'] =  pd.to_numeric(df['Bare Nuclei'], errors='coerce')

# Loại bỏ các bản ghi có giá trị thiếu
df = df.dropna().reset_index(drop=True)

# Phân tách đặc trưng và nhãn
X = df.drop('Class', axis=1)
y = df['Class']

# Chuyển lớp thành 0 (benign) và 1 (malignant) nếu cần
# Theo dataset: 2 = benign, 4 = malignant
y = y.map({2: 0, 4: 1})

# Chia train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.2, 
                                                    random_state=42)

## 3. Xây dựng và huấn luyện mô hình

In [5]:
# Khởi tạo các mô hình
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

# Huấn luyện và lưu kết quả
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, target_names=['Benign', 'Malignant'])
    cm = confusion_matrix(y_test, y_pred)
    results[name] = {'accuracy': acc, 'report': report, 'confusion_matrix': cm}

# Hiển thị kết quả
for name, res in results.items():
    print(f"Model: {name}")
    print(f"- Accuracy: {res['accuracy']:.4f}\n")
    print(res['report'])
    print("Confusion Matrix:")
    print(res['confusion_matrix'])
    print("\n" + "-"*50 + "\n")

Model: Logistic Regression
- Accuracy: 0.9562

              precision    recall  f1-score   support

      Benign       0.94      0.99      0.96        79
   Malignant       0.98      0.91      0.95        58

    accuracy                           0.96       137
   macro avg       0.96      0.95      0.95       137
weighted avg       0.96      0.96      0.96       137

Confusion Matrix:
[[78  1]
 [ 5 53]]

--------------------------------------------------

Model: Decision Tree
- Accuracy: 0.9343

              precision    recall  f1-score   support

      Benign       0.92      0.97      0.94        79
   Malignant       0.96      0.88      0.92        58

    accuracy                           0.93       137
   macro avg       0.94      0.93      0.93       137
weighted avg       0.94      0.93      0.93       137

Confusion Matrix:
[[77  2]
 [ 7 51]]

--------------------------------------------------

Model: Random Forest
- Accuracy: 0.9489

              precision    recall  f1

## 4. Kết luận

- So sánh độ chính xác và ma trận nhầm lẫn của các mô hình.
- Lựa chọn mô hình tốt nhất dựa trên accuracy và chi tiết của classification report.